In [81]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
import pickle

In [82]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [83]:
#preprocess the data
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [84]:
# Encode categorical variables
label_encoder_gender =LabelEncoder()
data['Gender'] =label_encoder_gender.fit_transform(data['Gender'])

In [85]:
onehot_encoder_geo=OneHotEncoder(handle_unknown='ignore')
geo_encoded=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [86]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [87]:
## Divide the dataset into indepent and dependent features
X=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']

##Split the data in training and testing sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

##Scale the features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [88]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

#### ANN regression problem statement

In [89]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [90]:
##Build oue Ann model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),## HL1 connected with input layer
    Dense(32,activation='relu'), ##HL2
    Dense(1) ##output layer

])
## compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])
model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_12 (Dense)            (None, 64)                832       
                                                                 
 dense_13 (Dense)            (None, 32)                2080      
                                                                 
 dense_14 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [91]:
## set up the tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

#setup Tensorboard
log_dir="regressionlogs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [92]:
#set up Early stopping 
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [93]:
#Train the model
history=model.fit(
    X_train,y_train,
    validation_data=(X_test,y_test),
    epochs=100,
    callbacks=[early_stopping_callback,tensorflow_callback]
    )

Epoch 1/100
250/250 [==============================] - 1s 2ms/step - loss: 100371.7734 - mae: 100371.7734 - val_loss: 98493.7578 - val_mae: 98493.7578
Epoch 2/100
250/250 [==============================] - 0s 1ms/step - loss: 99552.8672 - mae: 99552.8672 - val_loss: 96848.3594 - val_mae: 96848.3594
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 96703.5703 - mae: 96703.5703 - val_loss: 92672.5469 - val_mae: 92672.5469
Epoch 4/100
250/250 [==============================] - 0s 2ms/step - loss: 91122.0781 - mae: 91122.0781 - val_loss: 85752.2734 - val_mae: 85752.2734
Epoch 5/100
250/250 [==============================] - 0s 2ms/step - loss: 83074.9766 - mae: 83074.9766 - val_loss: 76987.7344 - val_mae: 76987.7344
Epoch 6/100
250/250 [==============================] - 0s 2ms/step - loss: 73783.3047 - mae: 73783.3047 - val_loss: 68004.4609 - val_mae: 68004.4609
Epoch 7/100
250/250 [==============================] - 0s 2ms/step - loss: 64924.2852 - mae: 64924.2852 

In [94]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [95]:
%tensorboard --logdir regressionlogs/fit20260523-175643

Reusing TensorBoard on port 6007 (pid 6632), started 1:04:05 ago. (Use '!kill 6632' to kill it.)

In [96]:
## Evaluate moodel on test data
test_loss,test_mae=model.evaluate(X_test,y_test)
print(f'Test MAE : {test_mae}')

63/63 [==============================] - 0s 4ms/step - loss: 50239.7812 - mae: 50239.7812
Test MAE : 50239.78125


In [97]:
model.save('regression_model.h5')

c:\Users\kotad\ANN end to end project\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
